# Eksperimen 4: XGBoost + Theta vs XGBoost + BiLSTM

Dengan **XGBoost** tahap-1, membandingkan **Theta** (tanpa neural) melawan **BiLSTM** sebagai koreksi residual.

In [ ]:
import sys
BASE='/home/mada/Documents/Kuliah/Algorithms and Complexity/02_Kode/hybrid-lgbm-xgb'
sys.path.insert(0,BASE)
import hybrid as H
from compare_ref_vs_gbm import evaluate, stage2_dispatch
import numpy as np, pandas as pd, re
import statsmodels.api as sm
# tambah dukungan theta pada dispatch (stage2_theta butuh resid test penuh)
def dispatch_with_theta(stage2, rt_s, re_s, horizon):
    if stage2 == 'theta':
        return H.stage2_theta(rt_s, re_s, horizon), None
    return stage2_dispatch(stage2, rt_s, re_s, horizon)
import compare_ref_vs_gbm as CR
CR.stage2_dispatch = dispatch_with_theta
# jalankan XGBoost+Theta (9 sel)
theta={}
for p in ['PM25','PM10','NO2']:
    for h in [1,7,30]:
        r=evaluate('xgboost','theta',h,p)
        if r: theta[(p,h)]=r['rmse']
        else: print('None:',p,h)
bilstm={}
for l in open(f'{BASE}/hasil_ref_vs_gbm.txt'):
    m=re.match(r'\[OK\] xgboost\+bilstm (\S+) h(\d+): RMSE=([\d.]+)',l)
    if m: bilstm[(m.group(1),int(m.group(2)))]=float(m.group(3))
rows=[]
for p in ['PM25','PM10','NO2']:
    for h in [1,7,30]:
        rows.append(dict(pol=p,hz=h,BiLSTM=bilstm[(p,h)],Theta=theta[(p,h)]))
print(pd.DataFrame(rows).set_index(['pol','hz']))


In [ ]:
import matplotlib.pyplot as plt
sub=pd.DataFrame(rows)
fig,ax=plt.subplots(figsize=(9,4))
x=np.arange(9); w=0.35
ax.bar(x-w,sub.BiLSTM,w,label='BiLSTM',color='green')
ax.bar(x+w,sub.Theta,w,label='Theta',color='red')
ax.set_xticks(x); ax.set_xticklabels([f'{r.pol} h{r.hz}' for _,r in sub.iterrows()],rotation=45,fontsize=8)
ax.set_ylabel('RMSE'); ax.set_title('XGBoost: BiLSTM vs Theta'); ax.legend(); ax.grid(alpha=0.3,axis='y')
plt.tight_layout(); plt.show()